# Segmenting Customers by How They Buy

1,589 customers pass through this store, and right now the business treats
them as one group: the same email cadence, the same discount thresholds, the
same priority in customer service. That wastes effort in both directions,
protecting a customer worth nothing the same way it protects one worth
thousands. This notebook groups customers by how they actually buy, not by how
large their running total happens to be, and hands the business a small set of
segments it can treat differently.

The original team ran this same exercise on raw per-customer sums of sales,
quantity and profit. A sum mostly measures tenure: how long someone has been a
customer and how many times they have ordered, so the clusters it produces
differ mainly in size, not behavior. This version replaces the sums with an
RFM-style feature set built from `olap.fact_order`, the order grain, since
customer behavior happens once per order and rolling it up from the order-line
grain would count a five-line order five times.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "utils").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
# Two harmless Windows/MKL environment quirks that otherwise print warnings
# on every KMeans fit below: a thread-count heuristic and a duplicate-OpenMP
# library check. Neither affects the results, both are silenced at the source
# rather than blanket-suppressed.
os.environ.setdefault("OMP_NUM_THREADS", "7")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

from utils import custom_plots as cp
from utils import custom_stats as cs
from utils.db_utils import run_query

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 0

## 1. One row per customer

`fact_order` already sits at the order grain, so this rolls orders up one
level further, to the customer. The reference date for recency is computed
inline as the latest order date anywhere in the table, not typed in as a
literal, so it stays correct if the warehouse is ever reloaded with a
different date range.

In [3]:
customers = run_query('''
    WITH bounds AS (
        SELECT MAX(od.full_date) AS dataset_end
        FROM olap.fact_order fo
        JOIN olap.dim_order_date od ON od.date_key = fo.order_date_key
    )
    SELECT
        c.customer_key,
        c.customer_id,
        c.segment,
        COUNT(*) AS frequency,
        b.dataset_end - MAX(od.full_date) AS recency_days,
        AVG(fo.sales)::float AS avg_order_value,
        AVG(fo.discount_rate)::float AS avg_discount_rate,
        (SUM(fo.profit) / NULLIF(SUM(fo.sales), 0))::float AS profit_margin,
        AVG(fo.quantity)::float AS avg_basket_size,
        SUM(fo.profit)::float AS total_profit,
        b.dataset_end AS dataset_end
    FROM olap.fact_order fo
    JOIN olap.dim_customer c ON c.customer_key = fo.customer_key
    JOIN olap.dim_order_date od ON od.date_key = fo.order_date_key
    CROSS JOIN bounds b
    GROUP BY c.customer_key, c.customer_id, c.segment, b.dataset_end
''')
customers.shape

(1589, 11)

In [4]:
print(f"dataset end (latest order date): {customers['dataset_end'].iloc[0]}")
print(f"missing values in any aggregated column: {customers.isna().any().any()}")

dataset end (latest order date): 2014-12-31
missing values in any aggregated column: False


1,589 customers, one row each, no missing values. `COUNT(*)` guarantees every
customer has at least one order behind their row, so none of the averages
below are dividing by zero.

## 2. Six features, not nine sums

A sum of sales is mostly a proxy for tenure: a customer who has ordered 30
times racks up a bigger total than one who has ordered 3 times even if their
per-order behavior is identical. Averaging strips tenure back out and leaves
the actual behavior: how much someone spends, discounts and profits per visit.
`profit_margin` is the one exception worth a note of its own: it is built as a
customer's total profit over total sales, not the mean of each order's own
margin, so one tiny order with a wild ratio does not carry the same weight as
a customer's large, representative orders.

The original also pivoted a count of orders per calendar year (2011 through
2014) onto the customer frame. That column encodes tenure twice over, since
recency and frequency already carry it, and it ties the model to whichever
four years the extract happens to span. It is left out here.

In [5]:
feature_audit = pd.DataFrame([
    {"feature": "recency_days", "definition": "days from a customer's last order to the dataset's latest order date",
     "why it's here": "flags who has drifted away"},
    {"feature": "frequency", "definition": "count of orders placed",
     "why it's here": "how often the relationship gets used"},
    {"feature": "avg_order_value", "definition": "mean of order-level sales",
     "why it's here": "spend per visit; a sum would just track tenure"},
    {"feature": "avg_discount_rate", "definition": "mean of order-level discount rate",
     "why it's here": "how much price-cutting it takes to keep them buying"},
    {"feature": "profit_margin", "definition": "total profit over total sales (dollar-weighted, not order-weighted)",
     "why it's here": "whether the relationship is actually profitable"},
    {"feature": "avg_basket_size", "definition": "mean of order-level quantity",
     "why it's here": "units per visit, a size dimension separate from dollar value"},
])
feature_audit

,feature,definition,why it's here
0,recency_days,days from a customer's last order to the datas...,flags who has drifted away
1,frequency,count of orders placed,how often the relationship gets used
2,avg_order_value,mean of order-level sales,spend per visit; a sum would just track tenure
3,avg_discount_rate,mean of order-level discount rate,how much price-cutting it takes to keep them b...
4,profit_margin,total profit over total sales (dollar-weighted...,whether the relationship is actually profitable
5,avg_basket_size,mean of order-level quantity,"units per visit, a size dimension separate fro..."


None of these is derived from another one on this list, so there is no
algebraic double-counting the way `cost` and `profit_margin` double-count
`profit` in the line-level data. A correlation check is still worth running,
since two features can be redundant in practice without being redundant by
formula.

In [6]:
raw_feature_cols = ["recency_days", "frequency", "avg_order_value",
                     "avg_discount_rate", "profit_margin", "avg_basket_size"]

cp.correlation_heatmap(
    customers[raw_feature_cols], method="pearson", show_significance=True,
    title="Correlation Among the Six Candidate Features",
)

The strongest pair is `avg_discount_rate` against `profit_margin` at -0.67:
customers who need deeper discounts to keep buying tend to run thinner
margins, which matches the plain intuition that heavier discounting erodes
margin. `avg_order_value` and `avg_basket_size` sit at 0.68, since a bigger
basket is usually also a pricier one. `recency_days` and `frequency` are at
-0.48: customers who order more often are, unsurprisingly, less likely to have
gone quiet recently. None of the six pairs is close to the 0.9-plus range that
would call for dropping one of them.

## 3. Skew, before scaling

K-means assumes roughly spherical clusters in the feature space it is given. A
heavily skewed feature stretches that space along one axis and pulls cluster
centers toward its outliers, so it is worth checking before anything gets
scaled.

In [7]:
skew_check = cs.summary_stats(customers, cols=raw_feature_cols)
skew_check[["column", "skew", "median", "mean", "min", "max"]].round(3)

,column,skew,median,mean,min,max
0,recency_days,3.007,36.000,87.655,0.000,1206.000
1,frequency,0.246,14.000,15.754,1.000,41.000
2,avg_order_value,1.237,412.717,426.546,7.173,1845.765
3,avg_discount_rate,1.334,0.142,0.152,0.000,0.700
4,profit_margin,-3.156,0.127,0.074,-2.170,0.451
5,avg_basket_size,0.409,6.294,6.009,1.000,27.000


`recency_days` is heavily right-skewed (skew 3.0): most customers ordered
recently and a long tail stretches out past a thousand days. `avg_order_value`
is skewed too (1.24), the usual shape for a spend variable, with a handful of
big spenders pulling the tail out. Both get a `log1p` transform before
scaling.

`avg_discount_rate` (1.33) and `profit_margin` (-3.16) are also skewed, but
neither is a log candidate: the discount rate is a bounded ratio that includes
true zeros, and `profit_margin` is negative for 290 customers, so a log of it
is not defined. Both stay on their original scale. Their skew will still show
up as elongated rather than perfectly round clusters, and that shows up
honestly in the silhouette score later rather than being hidden by a transform
that does not actually apply to them.

In [8]:
customers["recency_log"] = np.log1p(customers["recency_days"])
customers["monetary_log"] = np.log1p(customers["avg_order_value"])

## 4. Scaling

K-means clusters on Euclidean distance, and distance is a sum of squared
differences across features. Left on natural units, `avg_order_value` (tens to
low thousands of dollars) would swamp `avg_discount_rate` (0 to 0.7) and
decide the clusters almost by itself, with the other four features barely
moving the outcome. Standardizing every feature to mean 0 and standard
deviation 1 puts them on equal footing before K-means runs at all.

This is not an optional step to be compared against an unscaled run. The
original notebook fit a second model on raw, unscaled sums and reported it as
an alternative with a different k; on data with mismatched units that is not a
second opinion, it is a model dominated by whichever column happens to have
the largest range. It is not repeated here.

In [9]:
feature_cols = ["recency_log", "frequency", "monetary_log",
                "avg_discount_rate", "profit_margin", "avg_basket_size"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers[feature_cols])
X_scaled.shape

(1589, 6)

## 5. Choosing k

Two criteria, computed the same way at every k from 2 to 10: inertia
(within-cluster sum of squared distances, which always falls as k grows) and
the mean silhouette score (which has an actual maximum).

In [10]:
k_values = list(range(2, 11))
inertias, silhouettes = [], []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_k))

pd.DataFrame({"k": k_values, "inertia": inertias, "silhouette": silhouettes}).round(4)

D:\miniconda3\envs\analyst_313\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning:


Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md




,k,inertia,silhouette
0,2,6420.1144,0.3390
1,3,4910.9005,0.3461
2,4,4410.5290,0.3212
3,5,4005.3949,0.2913
4,6,3641.7942,0.2172
5,7,3405.2172,0.2190
6,8,3193.3167,0.2203
7,9,3032.6398,0.2216
8,10,2901.1497,0.1839


In [11]:
cp.elbow_plot(k_values, inertias, silhouettes, title="Choosing k for the Customer Segments")

The two criteria disagree. Inertia's elbow sits at k=4; the silhouette score
peaks at k=3 (0.346) and falls at every k after that. That disagreement is the
useful outcome, not a problem to explain away: it means the data has no single
obvious k, so the choice comes down to which one is more useful and more
repeatable.

In [12]:
def _ari_spread(k, n_seeds=5):
    labelings_k = [KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(X_scaled)
                   for s in range(n_seeds)]
    return [adjusted_rand_score(labelings_k[0], labelings_k[s]) for s in range(1, n_seeds)]

ari_k3 = _ari_spread(3)
ari_k4 = _ari_spread(4)
print(f"k=3, ARI vs. seed 0 across 4 other seeds: {[round(a, 3) for a in ari_k3]}")
print(f"k=4, ARI vs. seed 0 across 4 other seeds: {[round(a, 3) for a in ari_k4]}")

k=3, ARI vs. seed 0 across 4 other seeds: [1.0, 1.0, 1.0, 1.0]
k=4, ARI vs. seed 0 across 4 other seeds: [0.953, 0.997, 0.958, 0.93]


That settles it. k=3 reproduces perfectly across every seed tested; k=4 does
not, because the fourth cluster it adds is small and sits right on the
boundary between two others, and small boundary clusters are exactly what
shifts when K-means starts from a different point. Section 8 repeats this
check in more depth, but only on the model actually being kept.

k=3 is what gets fit below: the higher silhouette, and a segmentation a
manager could rebuild next quarter and get the same three groups back.

## 6. Fitting the chosen model

In [13]:
kmeans_final = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
customers["cluster"] = kmeans_final.fit_predict(X_scaled)

final_silhouette = silhouette_score(X_scaled, customers["cluster"])
print(f"silhouette score: {final_silhouette:.3f}")

silhouette score: 0.346


0.346. On customer RFM data that is a real but modest result: the segments are
separated well enough to be more useful than treating every customer the same,
not so well separated that they read as three naturally occurring species of
customer. Selling this as clean, discovered clusters would be overselling it.
Treated as a working set of business conventions built for a specific
decision, a discount and retention policy that currently makes none of these
distinctions, it holds up.

In [14]:
sil_samples = silhouette_samples(X_scaled, customers["cluster"])
cp.silhouette_plot(sil_samples, customers["cluster"], title="Per-Customer Fit Within Each Segment")

In [15]:
customers["silhouette"] = sil_samples
sil_by_cluster = customers.groupby("cluster")["silhouette"].agg(
    n="size", mean="mean", share_negative=lambda s: (s < 0).mean(),
)
sil_by_cluster["share_negative"] = (sil_by_cluster["share_negative"] * 100).round(1)
print(f"overall share with a negative silhouette: {(sil_samples < 0).mean():.1%}")
sil_by_cluster.round(3)

overall share with a negative silhouette: 4.4%


,n,mean,share_negative
cluster,,,
0,187,0.172,18.7
1,831,0.450,0.0
2,571,0.252,6.1


The three blades are not the same width. The largest cluster, 831 customers,
averages a silhouette of 0.45 with essentially no member scoring negative: it
is the cleanest, most internally consistent group of the three. The other two
average 0.17 and 0.25, with 18.7% and 6.1% of their members respectively
sitting closer to a neighboring cluster than their own. Across all 1,589
customers, 4.4% land with a negative silhouette overall. That is the honest
cost of the skew that could not be transformed away in Section 3, and it is
small enough not to change which group a manager would act on for any
individual customer.

## 7. Naming and profiling the segments

Names are assigned by ranking the clusters on the number that matters most to
the business, their share of total profit, rather than being picked by hand.

In [16]:
total_profit_all = customers["total_profit"].sum()
profit_share = customers.groupby("cluster")["total_profit"].sum() / total_profit_all
ranked_clusters = profit_share.sort_values(ascending=False).index.tolist()

segment_names = {
    ranked_clusters[0]: "Core Loyalists",
    ranked_clusters[1]: "Occasional Buyers",
    ranked_clusters[2]: "At-Risk Discounters",
}
customers["segment_name"] = customers["cluster"].map(segment_names)
segment_order = ["Core Loyalists", "Occasional Buyers", "At-Risk Discounters"]
segment_names

{1: 'Core Loyalists', 2: 'Occasional Buyers', 0: 'At-Risk Discounters'}

In [17]:
profile = customers.groupby("segment_name")[raw_feature_cols].median().round(2)
profile["n_customers"] = customers.groupby("segment_name").size()
profile["profit_share_pct"] = (
    customers.groupby("segment_name")["total_profit"].sum() / total_profit_all * 100
).round(1)
profile.loc[segment_order]

,recency_days,frequency,avg_order_value,avg_discount_rate,profit_margin,avg_basket_size,n_customers,profit_share_pct
segment_name,,,,,,,,
Core Loyalists,21.0,25.0,505.30,0.14,0.13,7.43,831,90.8
Occasional Buyers,95.0,6.0,286.43,0.12,0.17,4.14,571,14.3
At-Risk Discounters,121.0,5.0,154.43,0.31,-0.31,4.00,187,-5.2


**Core Loyalists**, 831 customers (52.3% of the base), carry 90.8% of the
store's total profit. They ordered 21 days ago at the median, order 25 times
over the data window, spend $505 a visit and run a healthy 13% margin on a
discount rate of just 14%. Half the customer base, essentially all the profit.

**Occasional Buyers**, 571 customers (35.9%), carry 14.3% of profit. They buy
less often (6 orders, last one 95 days ago) but at a slightly better margin
per order than the Loyalists, 17% against 13%, on a lighter discount. The
profit is there per order; the volume is not.

**At-Risk Discounters**, 187 customers (11.8%), carry -5.2% of profit: as a
group they are not a low-margin segment, they are a money-losing one. Median
discount rate is 31%, more than double the store-wide 14%, median margin is
-31%, and the median customer has not ordered in 121 days. The discounting is
not buying loyalty either; this is the group with the longest gap since its
last order.

In [18]:
at_risk_total = customers.loc[customers["segment_name"] == "At-Risk Discounters", "total_profit"].sum()
print(f"At-Risk Discounters' actual profit contribution across the dataset: ${at_risk_total:,.0f}")

At-Risk Discounters' actual profit contribution across the dataset: $-72,963


In [19]:
segment_means = customers.groupby("segment_name")[raw_feature_cols].mean().round(3)
segment_means.loc[segment_order]

,recency_days,frequency,avg_order_value,avg_discount_rate,profit_margin,avg_basket_size
segment_name,,,,,,
Core Loyalists,30.442,24.351,543.090,0.139,0.118,7.560
Occasional Buyers,139.802,6.583,329.370,0.115,0.163,4.305
At-Risk Discounters,182.674,5.551,205.373,0.324,-0.388,4.317


In [20]:
cp.radar_plot(
    customers, category_col="segment_name", value_cols=raw_feature_cols,
    title="Segment Profiles Across the Six Features",
)

The radar uses each segment's mean rather than the median in the table above,
since the plot aggregates by mean, and every axis is scaled so the lowest of
the three segments sits at the center and the highest sits at the outer edge.
Core Loyalists reach the outer edge on frequency, avg_order_value and
avg_basket_size, and sit close to it on profit_margin too: near-total
dominance on every dimension except how recently and how heavily they have
been discounted, where low is the good outcome. At-Risk Discounters are close
to the mirror image, sitting at the center on every axis except recency_days
and avg_discount_rate, where they are the only segment to reach the edge, the
combination of having gone quietest and being discounted hardest.

Occasional Buyers is the one place the ranking flips: their mean profit
margin, 16.3%, is the highest of the three segments, ahead of Core Loyalists
at 11.8%, the same gap the median table above already showed. A smaller,
quieter segment is the most profitable one per dollar of sales; it just does
not generate enough dollars to matter as much in total.

In [21]:
cp.cross_tab_heatmap(
    customers, "segment_name", "segment", normalize="row", show_counts=True,
    row_order=segment_order, colorscale=cp.SEQ_BLUE,
    title="Existing Consumer / Corporate / Home Office Mix Within Each Behavioral Segment",
)

Consumer, Corporate and Home Office split roughly 51% / 30% / 19% inside every
one of the three behavioral segments, the same mix as the customer base
overall. Buying behavior here is independent of the demographic segment
already sitting in `dim_customer`: this clustering is not just rediscovering a
field that already exists in the warehouse.

In [22]:
playbook = pd.DataFrame([
    {"segment": "Core Loyalists", "n_customers": int((customers["segment_name"] == "Core Loyalists").sum()),
     "profit_share_pct": profile.loc["Core Loyalists", "profit_share_pct"],
     "action": "Protect first. Priority service and loyalty perks, not further discounting; losing a handful of these customers costs more than the entire At-Risk segment is worth."},
    {"segment": "Occasional Buyers", "n_customers": int((customers["segment_name"] == "Occasional Buyers").sum()),
     "profit_share_pct": profile.loc["Occasional Buyers", "profit_share_pct"],
     "action": "Grow frequency. Margin per order is already good; re-engagement nudges (reminders, bundles) aimed at a second and third visit convert directly to profit without cutting price."},
    {"segment": "At-Risk Discounters", "n_customers": int((customers["segment_name"] == "At-Risk Discounters").sum()),
     "profit_share_pct": profile.loc["At-Risk Discounters", "profit_share_pct"],
     "action": "Cap the discount. The current discount depth is not preventing churn (121-day median recency) and is actively unprofitable; a lower discount ceiling loses little and stops the bleeding."},
])
playbook

,segment,n_customers,profit_share_pct,action
0,Core Loyalists,831,90.8,Protect first. Priority service and loyalty pe...
1,Occasional Buyers,571,14.3,Grow frequency. Margin per order is already go...
2,At-Risk Discounters,187,-5.2,Cap the discount. The current discount depth i...


## 8. Are these segments stable

K-means starts from a random set of centers. If a different random start gives
a different customer a different label, the segments are an artifact of that
start, not something to build a campaign on. Refitting the chosen k=3 model
from 10 different random seeds and comparing every pair with the adjusted Rand
index checks this directly.

In [23]:
labelings = []
for seed in range(10):
    km_seed = KMeans(n_clusters=3, n_init=10, random_state=seed)
    labelings.append(km_seed.fit_predict(X_scaled))

pairwise_ari = [
    adjusted_rand_score(labelings[i], labelings[j])
    for i in range(len(labelings)) for j in range(i + 1, len(labelings))
]
print(f"adjusted Rand index across 10 seeds, {len(pairwise_ari)} pairs: "
      f"min {min(pairwise_ari):.3f}, mean {np.mean(pairwise_ari):.3f}, max {max(pairwise_ari):.3f}")

adjusted Rand index across 10 seeds, 45 pairs: min 1.000, mean 1.000, max 1.000


Every pair of the 45 comparisons comes back at 1.000. This particular solution
is not just stable, it is fully deterministic across random starts on this
data: the three segments are a genuine structure in the feature space, not a
coin flip that happened to land the same way once.

## 9. A two-dimensional look

The original used t-SNE and called the resulting picture evidence that the
data was "clusterable." It is not: a t-SNE layout is a function of its
perplexity setting and will produce visually distinct blobs on data with no
real structure at all, so it cannot answer that question either way. What
follows is a PCA projection of the segments already fit above, useful only for
looking at the result, not for judging whether clusters exist.

In [24]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
print(f"variance captured: PC1 {pca.explained_variance_ratio_[0]:.1%}, "
      f"PC2 {pca.explained_variance_ratio_[1]:.1%}, "
      f"total {pca.explained_variance_ratio_.sum():.1%}")

pca_view = pd.DataFrame({"pc1": coords[:, 0], "pc2": coords[:, 1],
                          "segment_name": customers["segment_name"]})

variance captured: PC1 45.5%, PC2 24.2%, total 69.7%


In [25]:
cp.scatter_plot(
    pca_view, x="pc1", y="pc2", color_by="segment_name", trendline=False,
    title="Customer Segments Projected Onto Two Principal Components",
)

Two components hold 69.7% of the variance in the six scaled features, so the
picture is a reasonable summary without being the whole story. Core Loyalists
form a distinct mass on the right; Occasional Buyers and At-Risk Discounters
overlap noticeably in the middle, which is the same story the 0.25 and 0.17
silhouette averages already told in Section 6, visible here instead of just
measured.

## 10. What this means for the business

The decision this model changes is which list a customer's ID lands on: the
protect list, the grow list, or the discount-cap list, each with a different
budget and a different discount authority attached to it. Getting a customer
on the wrong list has an asymmetric cost. A Core Loyalist mistakenly flagged
as At-Risk gets an unnecessary discount that eats straight into an already
thin 13% margin, repeated across 831 customers if the mistake is systematic.
An At-Risk customer left on the current policy keeps costing the store money
on every order, the same way the 187 customers in that segment already have.

Three findings carry a number a manager can act on directly. Just over half
the customer base, the Core Loyalists, produces 90.8% of total profit, so the
single highest-value action available is not acquiring new customers, it is
not losing these ones. 187 customers, 11.8% of the base, the At-Risk
Discounters, are a net drag on profit at the current discount policy, not
merely a weak performer, and capping their discount depth costs the business
little because that depth is not buying loyalty back (121-day median recency)
in the first place. The segmentation itself reproduces perfectly across 10
random seeds and sits apart from the existing Consumer/Corporate/Home Office
field, so it is a genuinely new axis to run a campaign against, not a
relabeling of something the business already tracks.

The At-Risk Discounters finding is corroborated elsewhere in the project
rather than resting on this clustering alone. `hypothesis_testing` shows that
a discount raises units sold by a negligible amount, and that the apparent
lift is mostly a market-mix effect. `ml/profit_regression` shows discount
depth as the strongest driver of loss-making lines at order-line grain. This
notebook adds the customer view: heavier discounting tracks worse margin
(Spearman r = -0.55) and does not track buying more often (r = -0.08). The
same conclusion arrived at three ways is worth more than any one of them.